In [ ]:
# 1. Remove mismatched dependencies that trigger SIGSEGV (-11)
!pip uninstall -y torchaudio torchvision torchcodec jax jaxlib

# 2. Install matching Torch & PyTorch/XLA 2.8.0
!pip install torch==2.8.0 torch_xla[tpu]==2.8.0 -f https://storage.googleapis.com/libtpu-releases/index.html --quiet

# 3. Install core fine-tuning libraries (do NOT install torch_xla[pallas] or upgrade libtpu)
!pip install transformers==5.13.1 peft==0.20.0 accelerate==1.14.0 datasets --quiet

In [ ]:
import os
import logging

# --- 1. SILENCE JAX/PALLAS WARNING SPAM ---
class SuppressPallasWarning(logging.Filter):
    def filter(self, record):
        return "jax/pallas" not in record.getMessage()

logging.getLogger().addFilter(SuppressPallasWarning())
for handler in logging.root.handlers:
    handler.addFilter(SuppressPallasWarning())

# --- 2. ENFORCE ZERO CPU FALLBACK ---
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_FALLBACK_TO_CPU"] = "0"    # Fatal error immediately if any op leaves the TPU
os.environ["XLA_METRICS_LEVEL"] = "1"

import torch
import numpy as np
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
import torch_xla.distributed.spmd as xs
from torch_xla.experimental.spmd_fully_sharded_data_parallel import SpmdFullyShardedDataParallel as FSDPv2

# Patch torch_xla for device resolution
torch.xla = torch_xla

# Initialize SPMD on all 8 TPU v5e cores
xr.use_spmd()
device = xm.xla_device()

num_devices = xr.global_runtime_device_count()
print(f"Number of TPU devices visible: {num_devices}")
assert num_devices == 8, f"Expected 8 devices on v5e-8, found {num_devices}"

mesh_shape = (num_devices, 1)
device_ids = np.array(range(num_devices))
mesh = xs.Mesh(device_ids, mesh_shape, ('fsdp', 'model'))
print("SPMD Mesh initialized successfully (8-chip FSDP axis).")

In [ ]:
import math
import torch
import torch.nn.functional as F
import torch_xla
from torch_xla.distributed.fsdp import checkpoint_module
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from transformers.models.lfm2.modeling_lfm2 import Lfm2DecoderLayer, ALL_ATTENTION_FUNCTIONS

MODEL_ID = "LiquidAI/LFM2.5-2.6B"

# --- 1. Memory-Efficient Chunked Causal Attention ---
def chunked_causal_attention_forward(
    module,
    query_states,
    key_states,
    value_states,
    attention_mask=None,
    dropout=0.0,
    scaling=None,
    **kwargs
):
    """
    Computes causal attention in sequence chunks of 512 to prevent
    allocating the massive [32, 8192, 8192] FP32 matrix in TPU HBM.
    """
    batch_size, num_q_heads, q_len, head_dim = query_states.shape
    num_kv_heads = key_states.shape[1]
    
    # Handle GQA (repeat KV heads if mismatched)
    if num_q_heads != num_kv_heads:
        n_rep = num_q_heads // num_kv_heads
        key_states = torch.repeat_interleave(key_states, n_rep, dim=1)
        value_states = torch.repeat_interleave(value_states, n_rep, dim=1)
        
    scale = scaling if scaling is not None else (1.0 / math.sqrt(head_dim))
    query_states = query_states * scale

    CHUNK_SIZE = 512
    output_chunks = []

    for i in range(0, q_len, CHUNK_SIZE):
        q_chunk = query_states[:, :, i:i+CHUNK_SIZE, :]
        cur_q_len = q_chunk.shape[2]
        
        k_past = key_states[:, :, :i+cur_q_len, :]
        v_past = value_states[:, :, :i+cur_q_len, :]
        
        attn_weights = torch.matmul(q_chunk, k_past.transpose(-1, -2))
        
        row_indices = torch.arange(i, i + cur_q_len, device=query_states.device).unsqueeze(1)
        col_indices = torch.arange(0, i + cur_q_len, device=query_states.device).unsqueeze(0)
        causal_mask = col_indices <= row_indices
        
        attn_weights = torch.where(causal_mask, attn_weights, -10000.0)
        attn_probs = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        chunk_out = torch.matmul(attn_probs, v_past)
        output_chunks.append(chunk_out)

    attn_output = torch.cat(output_chunks, dim=2)
    attn_output = attn_output.transpose(1, 2).contiguous()
    attn_output = attn_output.reshape(batch_size, q_len, -1)
    
    return attn_output, None

# Register chunked attention interface
ALL_ATTENTION_FUNCTIONS.register("chunked_xla", chunked_causal_attention_forward)

print("Loading tokenizer and model in bfloat16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)

model.config._attn_implementation = "chunked_xla"
model.config.use_cache = False
model.gradient_checkpointing_disable()

# LoRA Setup (covers conv layers and attention/mlp layers)
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "w1", "w2", "w3"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

model = model.to(device)

# --- 2. Wrap all 30 layers with Native XLA checkpoint_module + FSDPv2 ---
wrapped_count = 0
for name, module in list(model.named_modules()):
    if isinstance(module, Lfm2DecoderLayer):
        parent_name = ".".join(name.split(".")[:-1])
        child_name = name.split(".")[-1]
        parent = model.get_submodule(parent_name)
        
        # Native Google XLA activation checkpointing + SPMD sharding
        ckpt_layer = checkpoint_module(module)
        sharded_layer = FSDPv2(ckpt_layer, mesh=mesh)
        
        setattr(parent, child_name, sharded_layer)
        wrapped_count += 1

print(f"Successfully wrapped {wrapped_count} layers with Google XLA checkpoint_module + FSDPv2")

In [ ]:
import os
import torch
from datasets import load_from_disk, Dataset
from torch.utils.data import DataLoader
import torch_xla.distributed.parallel_loader as pl

# ==============================================================================
# DATASET CONFIGURATION
# Set DATASET_PATH to your pre-tokenized arrow dataset directory on Kaggle.
# If left empty or path not found, it runs in DEMO MODE with synthetic data so
# anyone can test the pipeline immediately with zero external dependencies.
# ==============================================================================
DATASET_PATH = ""  # e.g., "/kaggle/input/your-dataset/tokenized"
MAX_LEN = 8192
BATCH_SIZE = 8     # 1 sample per chip across the 8-chip mesh

# Fast static tensor collator (zero Python list loops)
def fast_collate_fn(batch):
    b_size = len(batch)
    input_ids = torch.zeros((b_size, MAX_LEN), dtype=torch.long)
    labels = torch.full((b_size, MAX_LEN), -100, dtype=torch.long)
    attention_mask = torch.zeros((b_size, MAX_LEN), dtype=torch.long)
    
    for i, b in enumerate(batch):
        seq_len = min(len(b["input_ids"]), MAX_LEN)
        input_ids[i, :seq_len] = torch.tensor(b["input_ids"][:seq_len], dtype=torch.long)
        labels[i, :seq_len] = torch.tensor(b["labels"][:seq_len], dtype=torch.long)
        attention_mask[i, :seq_len] = torch.tensor(b["attention_mask"][:seq_len], dtype=torch.long)
        
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

if DATASET_PATH and os.path.exists(DATASET_PATH):
    print(f"Loading user dataset from: {DATASET_PATH}...")
    dataset = load_from_disk(DATASET_PATH)
    print(f"Loaded {len(dataset)} examples.")
else:
    print("No custom dataset provided or path not found.")
    print("Running in DEMO MODE with synthetic 8192-token sequences...")
    dataset = Dataset.from_dict({
        "input_ids": [list(range(100, 100 + 4000))] * 80,
        "labels": [[-100]*1000 + list(range(1100, 4100))] * 80,
        "attention_mask": [[1] * 4000] * 80,
    })

train_loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=fast_collate_fn, 
    drop_last=True,
    num_workers=2,
    persistent_workers=True
)
device_loader = pl.MpDeviceLoader(train_loader, device)
print("DataLoader and MpDeviceLoader initialized successfully.")

In [ ]:
import os
import time
import torch
import torch_xla.distributed.spmd as xs
import torch_xla.core.xla_model as xm
import torch_xla.debug.metrics as met
from peft import get_peft_model_state_dict

# Training parameters
NUM_STEPS = 20      # Set to len(train_loader) for a full epoch, or a test number like 20
LEARNING_RATE = 2e-4
LOG_EVERY = 1
SAVE_DIR = "/kaggle/working/lfm25_lora_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
model.train()

print(f"Starting training on 8 TPU cores (Target steps: {NUM_STEPS})...\n", flush=True)

step_start = time.time()
for step, batch in enumerate(device_loader):
    if step >= NUM_STEPS:
        break

    # Shard inputs along the FSDP axis
    for key in batch:
        xs.mark_sharding(batch[key], mesh, ('fsdp', None))

    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    xm.mark_step()

    step_duration = time.time() - step_start
    step_start = time.time()

    if step == 0:
        print(f"Step 1 compiled & executed in {step_duration:.1f}s | Loss: {loss.item():.4f}", flush=True)
    elif (step + 1) % LOG_EVERY == 0:
        print(f"Step {step+1}/{NUM_STEPS} took {step_duration:.2f}s | Loss: {loss.item():.4f}", flush=True)

# --- SAFE LORA SAVE TO DISK ---
print("\nSaving LoRA adapter to disk...", flush=True)
lora_state_dict = get_peft_model_state_dict(model)

# Move tensors from TPU virtual storage to host CPU memory before saving
cpu_lora_state_dict = {k: v.cpu() for k, v in lora_state_dict.items()}

model.save_pretrained(SAVE_DIR, state_dict=cpu_lora_state_dict)
tokenizer.save_pretrained(SAVE_DIR)
print(f"--> SUCCESS: LoRA adapter cleanly saved to {SAVE_DIR}!", flush=True)

# Verify zero CPU fallbacks occurred
report = met.metrics_report()
if "UnloweredOpCount" in report:
    print("\nWARNING: CPU fallbacks detected during training:")
    print(report)
else:
    print("\nAUDIT PASSED: 0 CPU Fallbacks detected! All ops ran natively on TPU v5e-8.")